# Getting Started: A Minimal Example

This tutorial walks through a complete Jim run on a toy problem.
We inject a gravitational-wave signal, fix every parameter except
chirp mass ($\mathcal{M}_c$) and luminosity distance ($d_L$), and
sample just those two to keep things fast and easy to visualise.

## Setup

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)

from jimgw.core.single_event.detector import get_H1, get_L1
from jimgw.core.single_event.waveform import RippleIMRPhenomD
from jimgw.core.single_event.likelihood import TransientLikelihoodFD
from jimgw.core.single_event.gps_times import (
    greenwich_mean_sidereal_time as compute_gmst,
)
from jimgw.core.prior import CombinePrior, UniformPrior
from jimgw.core.jim import Jim

## 1. Define injection parameters

We pick a simple aligned-spin BBH signal. Only `M_c` and `d_L` will
be sampled; every other parameter is fixed to the injection value.

In [ ]:
gps_time = 1126259462.0
gmst = compute_gmst(gps_time)

injection_parameters = {
    "M_c": 28.0,
    "eta": 0.24,
    "s1_z": 0.0,
    "s2_z": 0.0,
    "d_L": 440.0,
    "t_c": 0.0,
    "phase_c": 0.0,
    "iota": 0.4,
    "psi": 0.3,
    "ra": 1.5,
    "dec": 0.5,
    "trigger_time": gps_time,
    "gmst": gmst,
}

## 2. Set up detectors and inject the signal

In [ ]:
waveform = RippleIMRPhenomD(f_ref=20.0)

f_min = 20.0
f_max = 1024.0
duration = 4.0
sampling_frequency = 2 * f_max

ifos = [get_H1(), get_L1()]
for ifo in ifos:
    ifo.load_and_set_psd()
    ifo.frequency_bounds = (f_min, f_max)
    ifo.inject_signal(
        duration,
        sampling_frequency,
        0.0,
        waveform,
        injection_parameters,
        is_zero_noise=False,
    )

## 3. Define the prior

We only sample `M_c` and `d_L`, so we only need priors for those two.

In [ ]:
prior = CombinePrior([
    UniformPrior(20.0, 40.0, ["M_c"]),
    UniformPrior(100.0, 1000.0, ["d_L"]),
])

## 4. Build the likelihood

All parameters that are not being sampled must be baked into the
likelihood via `fixed_parameters`.

In [ ]:
fixed_parameters = {
    k: v
    for k, v in injection_parameters.items()
    if k not in ["M_c", "d_L", "trigger_time", "gmst"]
}

likelihood = TransientLikelihoodFD(
    ifos,
    waveform=waveform,
    trigger_time=gps_time,
    f_min=f_min,
    f_max=f_max,
    fixed_parameters=fixed_parameters,
)

## 5. Sample with Jim

With only two parameters, we can use relatively few chains and loops.

In [ ]:
jim = Jim(
    likelihood,
    prior,
    n_chains=200,
    n_local_steps=50,
    n_global_steps=200,
    n_training_loops=5,
    n_production_loops=5,
    verbose=True,
)

jim.sample()

## 6. Inspect the results

In [ ]:
samples = jim.get_samples()